# GPU benchmark: spatial vs dual-domain U-Net

Run this notebook in Google Colab with **Runtime > Change runtime type > T4 GPU**. It mounts Google Drive, loads both `.pt` checkpoints, runs the same preprocessing and full-volume comparison used by the demo backend, and reports timing for each stage.

Put the following files in one Google Drive folder, or update the paths in the configuration cell:
- `baseline_best.pt`
- `dual_best.pt`
- one T2w file (`.nii` or `.nii.gz`)
- optional matching segmentation file for Dice scores

In [ ]:
!pip -q install nibabel matplotlib

import os
import time
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from google.colab import drive

drive.mount('/content/drive')

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
else:
    print('WARNING: running on CPU. Select a GPU runtime in Colab.')

In [ ]:
# Edit only this cell. BASE_DIR should contain both checkpoints and your MRI files.
BASE_DIR = '/content/drive/MyDrive/brain_tumor_models'
BASELINE_CKPT = os.path.join(BASE_DIR, 'baseline_best.pt')
DUAL_CKPT = os.path.join(BASE_DIR, 'dual_best.pt')
T2W_PATH = ''       # e.g. os.path.join(BASE_DIR, 'case-t2w.nii.gz')
SEG_PATH = ''       # optional; leave empty to skip Dice scores
BATCH_SIZE = 16     # try 32 or 64 if GPU memory allows
THRESHOLD = 0.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Auto-discover inputs when paths are left blank.
if not T2W_PATH:
    candidates = [p for p in glob.glob(os.path.join(BASE_DIR, '**', '*.nii*'), recursive=True)
                  if 'seg' not in os.path.basename(p).lower()]
    if candidates:
        T2W_PATH = candidates[0]
if not SEG_PATH:
    candidates = glob.glob(os.path.join(BASE_DIR, '**', '*seg*.nii*'), recursive=True)
    SEG_PATH = candidates[0] if candidates else ''

for path in (BASELINE_CKPT, DUAL_CKPT, T2W_PATH):
    print(path, 'OK' if path and os.path.exists(path) else 'MISSING')
print('Segmentation:', SEG_PATH if SEG_PATH else 'not provided')
print('Device:', DEVICE, '| Batch size:', BATCH_SIZE)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class Encoder(nn.Module):
    def __init__(self, in_ch=1, base=64):
        super().__init__()
        self.enc1, self.enc2 = ConvBlock(in_ch, base), ConvBlock(base, base*2)
        self.enc3, self.enc4 = ConvBlock(base*2, base*4), ConvBlock(base*4, base*8)
        self.bottleneck = ConvBlock(base*8, base*16)
        self.pool, self.drop = nn.MaxPool2d(2), nn.Dropout2d(0.3)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2)); e4 = self.enc4(self.pool(e3))
        b = self.drop(self.bottleneck(self.pool(e4)))
        return b, (e1, e2, e3, e4)

class Decoder(nn.Module):
    def __init__(self, bottleneck_ch=1024, base=64, out_ch=3):
        super().__init__()
        self.up4, self.up3 = nn.ConvTranspose2d(bottleneck_ch, base*8, 2, 2), nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.up2, self.up1 = nn.ConvTranspose2d(base*4, base*2, 2, 2), nn.ConvTranspose2d(base*2, base, 2, 2)
        self.dec4, self.dec3 = ConvBlock(base*16, base*8), ConvBlock(base*8, base*4)
        self.dec2, self.dec1 = ConvBlock(base*4, base*2), ConvBlock(base*2, base)
        self.drop, self.out = nn.Dropout2d(0.3), nn.Conv2d(base, out_ch, 1)
    def forward(self, b, skips):
        e1, e2, e3, e4 = skips
        d = self.drop(self.dec4(torch.cat([self.up4(b), e4], 1)))
        d = self.drop(self.dec3(torch.cat([self.up3(d), e3], 1)))
        d = self.dec2(torch.cat([self.up2(d), e2], 1))
        return self.out(self.dec1(torch.cat([self.up1(d), e1], 1)))

class SpatialUNet(nn.Module):
    def __init__(self, base=64, out_ch=3):
        super().__init__(); self.encoder = Encoder(1, base); self.decoder = Decoder(base*16, base, out_ch)
    def forward(self, x):
        b, skips = self.encoder(x); return self.decoder(b, skips)

class DualDomainUNet(nn.Module):
    def __init__(self, base=64, out_ch=3):
        super().__init__(); self.spatial_enc = Encoder(1, base); self.freq_enc = Encoder(1, base)
        self.fusion = nn.Sequential(nn.Conv2d(base*32, base*16, 1), nn.BatchNorm2d(base*16), nn.ReLU(inplace=True))
        self.decoder = Decoder(base*16, base, out_ch)
    def forward(self, spatial, freq):
        sb, skips = self.spatial_enc(spatial); fb, _ = self.freq_enc(freq)
        return self.decoder(self.fusion(torch.cat([sb, fb], 1)), skips)

In [ ]:
def load_model(model_class, path):
    model = model_class().to(DEVICE)
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    state = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state)
    model.eval()
    return model

def preprocess_volume(path):
    volume = nib.load(path).get_fdata().astype(np.float32)
    mask = volume > 0
    mean = volume[mask].mean() if mask.any() else 0.0
    std = volume[mask].std() if mask.any() else 1.0
    normalized = (volume - mean) / (std + 1e-8)
    slices = [z for z in range(normalized.shape[2]) if normalized[:, :, z].max() > 0]
    return normalized, slices

def make_inputs(volume, slices):
    spatial = np.transpose(volume[:, :, slices], (2, 0, 1)).astype(np.float32)
    kspace = np.fft.fftshift(np.fft.fft2(spatial, axes=(-2, -1)), axes=(-2, -1))
    magnitude = np.log1p(np.abs(kspace)).astype(np.float32)
    low = magnitude.min(axis=(-2, -1), keepdims=True)
    high = magnitude.max(axis=(-2, -1), keepdims=True)
    frequency = (magnitude - low) / (high - low + 1e-8)
    return torch.from_numpy(spatial[:, None]).to(DEVICE), torch.from_numpy(frequency[:, None]).to(DEVICE)

@torch.inference_mode()
def run_model(model, model_type, spatial, frequency, slices, shape):
    predictions = np.zeros((*shape, 3), dtype=np.float32)
    for start in range(0, len(slices), BATCH_SIZE):
        end = start + BATCH_SIZE
        if model_type == 'spatial': logits = model(spatial[start:end])
        else: logits = model(spatial[start:end], frequency[start:end])
        binary = (torch.sigmoid(logits) > THRESHOLD).cpu().numpy().astype(np.float32)
        for index, z in enumerate(slices[start:end]):
            predictions[:, :, z, :] = binary[index].transpose(1, 2, 0)
    if DEVICE.type == 'cuda': torch.cuda.synchronize()
    return predictions

def dice(pred, target, channel):
    p, t = pred[..., channel].ravel(), target[..., channel].ravel()
    return float((2 * (p * t).sum() + 1e-5) / (p.sum() + t.sum() + 1e-5))

def segmentation_mask(path):
    seg = nib.load(path).get_fdata().astype(np.int16)
    return np.stack([(seg >= 1), ((seg == 1) | (seg == 3)), (seg == 3)], axis=-1).astype(np.float32)

In [ ]:
assert os.path.exists(BASELINE_CKPT), f'Missing {BASELINE_CKPT}'
assert os.path.exists(DUAL_CKPT), f'Missing {DUAL_CKPT}'
assert os.path.exists(T2W_PATH), f'Missing {T2W_PATH}'

if DEVICE.type == 'cuda': torch.cuda.empty_cache(); torch.cuda.synchronize()
load_start = time.perf_counter()
baseline_model = load_model(SpatialUNet, BASELINE_CKPT)
dual_model = load_model(DualDomainUNet, DUAL_CKPT)
load_seconds = time.perf_counter() - load_start

prep_start = time.perf_counter()
volume, tissue_slices = preprocess_volume(T2W_PATH)
spatial_inputs, frequency_inputs = make_inputs(volume, tissue_slices)
if DEVICE.type == 'cuda': torch.cuda.synchronize()
prep_seconds = time.perf_counter() - prep_start

baseline_start = time.perf_counter()
baseline_predictions = run_model(baseline_model, 'spatial', spatial_inputs, frequency_inputs, tissue_slices, volume.shape)
baseline_seconds = time.perf_counter() - baseline_start

dual_start = time.perf_counter()
dual_predictions = run_model(dual_model, 'dual', spatial_inputs, frequency_inputs, tissue_slices, volume.shape)
dual_seconds = time.perf_counter() - dual_start

total_seconds = load_seconds + prep_seconds + baseline_seconds + dual_seconds
print(f'Volume: {volume.shape}; tissue slices: {len(tissue_slices)}')
print(f'Model loading:     {load_seconds:.2f} s')
print(f'Preprocessing:     {prep_seconds:.2f} s')
print(f'Baseline inference:{baseline_seconds:.2f} s')
print(f'Dual inference:    {dual_seconds:.2f} s')
print(f'Total:             {total_seconds:.2f} s')
print(f'Dual / baseline:   {dual_seconds / max(baseline_seconds, 1e-9):.2f}x')

In [ ]:
if SEG_PATH and os.path.exists(SEG_PATH):
    gt = segmentation_mask(SEG_PATH)
    names = ['WT', 'TC', 'ET']
    print('Dice scores')
    for channel, name in enumerate(names):
        print(f'{name}: baseline={dice(baseline_predictions, gt, channel):.4f}, dual={dice(dual_predictions, gt, channel):.4f}')
else:
    gt = None
    print('No segmentation file supplied; skipping Dice scores.')

counts = dual_predictions[:, :, :, 0].sum(axis=(0, 1))
z = int(np.argmax(counts)) if counts.max() > 0 else volume.shape[2] // 2
def overlay(image, mask):
    image = image - image.min(); image = image / (image.max() + 1e-8)
    result = np.stack([image] * 3, axis=-1)
    colors = np.array([[1.0, .42, .42], [.31, .80, .77], [1.0, .90, .43]])
    for c in range(3):
        selected = mask[:, :, c] > 0
        result[selected] = .55 * result[selected] + .45 * colors[c]
    return result

images = [volume[:, :, z], overlay(volume[:, :, z], baseline_predictions[:, :, z]), overlay(volume[:, :, z], dual_predictions[:, :, z])]
titles = [f'T2w, z={z}', 'Baseline spatial U-Net', 'Dual-domain U-Net']
fig, axes = plt.subplots(1, 4 if gt is not None else 3, figsize=(16 if gt is not None else 12, 4))
axes = np.atleast_1d(axes)
for axis, image, title in zip(axes, images, titles): axis.imshow(image, cmap='gray' if image.ndim == 2 else None); axis.set_title(title); axis.axis('off')
if gt is not None: axes[3].imshow(overlay(volume[:, :, z], gt[:, :, z])); axes[3].set_title('Ground truth'); axes[3].axis('off')
plt.tight_layout(); plt.show()

## Benchmark notes

- The first run includes model loading and CUDA kernel setup. For a fair inference comparison, run the inference cell again and compare the baseline and dual timings.
- Increase `BATCH_SIZE` until GPU memory becomes constrained. The dual-domain model uses substantially more memory.
- The reported total includes model loading, preprocessing, and both model inferences, but not notebook installation or Drive mount time.
- If the checkpoint folder is elsewhere, change only `BASE_DIR`; explicit `T2W_PATH` and `SEG_PATH` override auto-discovery.